# Aula 11 - Protocolo de Contexto de Modelo (MCP)

O **Protocolo de Contexto de Modelo (MCP)** é um padrão aberto que permite aos agentes descobrir e utilizar ferramentas, recursos e fontes de dados dinamicamente em tempo de execução. Em vez de codificar ferramentas diretamente num agente, o MCP permite que os agentes se conectem a servidores externos que expõem capacidades sob demanda.

Nesta aula, irá aprender:
- O que é o MCP e porque é importante para sistemas de agentes
- Como funciona a arquitetura cliente-servidor do MCP
- Como construir agentes que utilizam descoberta de ferramentas ao estilo MCP


## Configuração

**Pré-requisitos:**
- Projeto Microsoft Foundry com um modelo implementado
- Execute `az login` para autenticação


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## O que é o Protocolo de Contexto do Modelo (MCP)?

O MCP define uma forma padrão para agentes de IA descobrirem e interagirem com ferramentas externas e fontes de dados:

- **Servidor MCP**: Expõe ferramentas, recursos e prompts através de um protocolo padrão
- **Cliente MCP**: O runtime do agente que se conecta a servidores e descobre as capacidades disponíveis
- **Descoberta Dinâmica**: Os agentes não precisam de ferramentas codificadas — eles descobrem o que está disponível em tempo de execução

Isto é poderoso para construir sistemas de agentes extensíveis onde novas capacidades podem ser adicionadas sem modificar o código do agente.


## Como funciona o MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. O agente (cliente MCP) liga-se a um servidor MCP
2. O servidor responde com uma lista de ferramentas disponíveis e seus esquemas
3. O agente pode então chamar qualquer ferramenta descoberta durante o seu raciocínio
4. Os resultados retornam através do mesmo protocolo


## Simulando a Descoberta de Ferramentas MCP

Como um servidor MCP real requer um processo de servidor em execução, vamos demonstrar o padrão utilizando funções `@tool` que simulam o que um serviço de alojamento ligado a MCP forneceria.

Em produção, estas ferramentas seriam descobertas dinamicamente a partir de um servidor MCP em vez de definidas localmente.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Construir um Agente com Ferramentas ao Estilo MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP em Produção

Num ambiente de produção, o MCP permite padrões poderosos:

- **Descoberta dinâmica de ferramentas**: Agentes conectam-se a servidores MCP e descobrem ferramentas em tempo de execução
- **Arquitetura desacoplada**: Provedores de ferramentas podem atualizar independentemente do agente
- **Partilha entre organizações**: Equipas podem expor capacidades via servidores MCP que qualquer agente pode usar
- **Suporte ao Microsoft Agent Framework**: O MAF tem suporte cliente MCP incorporado via a integração `mcp`

Para usar um servidor MCP real com o MAF, deve conectar-se através de `hosted_mcp_tool()` ou a integração do cliente MCP.

**Saiba mais:**
- [Especificação MCP](https://modelcontextprotocol.io/)
- [Suporte MCP no Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Resumo

Nesta lição, aprendeu:
- **MCP** é um padrão aberto para descoberta dinâmica de ferramentas entre agentes e fornecedores de ferramentas
- A **arquitetura cliente-servidor** permite que agentes descubram capacidades em tempo de execução
- O MCP permite **sistemas de agentes extensíveis e desacoplados** onde ferramentas podem ser adicionadas sem alterações de código
- O Microsoft Agent Framework fornece **suporte MCP integrado** para uso em produção


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
